<a href="https://www.kaggle.com/code/mahdimashayekhi/mnist-classifier?scriptVersionId=268272527" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5))
])

training_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
validation_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)

training_loader = DataLoader(training_dataset, batch_size=64, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=64, shuffle=False)

In [ ]:
class Classifier(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()

        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

In [ ]:
device = ('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device is {device}.')

In [ ]:
input_size = 28 * 28
hidden_size = 128
output_size = 10
learning_rate = 0.001
num_epochs = 15 

model = Classifier(input_size, hidden_size, output_size).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
# Training Loop
loss_history = []
accuracy_history = []
for epoch in range(num_epochs):
    
    model.train()
    running_loss = 0.0
    running_accuracy = 0.0
    for images, labels in training_loader:
        images, labels = images.to(device), labels.to(device)

        images = images.view(images.shape[0], -1)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)

        running_loss += loss.item()
        running_accuracy += torch.sum(preds == labels.data)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    else:
        loss_history.append(running_loss)
        accuracy_history.append(running_accuracy)
        
        print(f'\nEpoch: {epoch+1}/{num_epochs}')
        print(f'Loss: {running_loss/len(training_loader.dataset):.4f}, Accuracy: {running_accuracy/len(training_loader.dataset) * 100:.4f} %')

In [ ]:
# Validation

print('Validation...')

val_loss_history = []
val_accuracy_history = []
for epoch in range(num_epochs):
    model.eval()
    val_running_loss = 0.0
    val_running_accuracy = 0.0

    with torch.no_grad():
        for images, labels in validation_loader:
            images, labels = images.to(device), labels.to(device)

            images = images.view(images.shape[0], -1)
            outputs = model(images)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

            val_running_loss += loss.item()
            val_running_accuracy += torch.sum(preds == labels.data)

        else:
            val_loss_history.append(val_running_loss)
            val_accuracy_history.append(val_running_accuracy)
            
            print(f'\nEpoch: {epoch+1}/{num_epochs}')
            print(f'Loss: {val_running_loss/len(validation_loader.dataset):.4f}, Accuracy: {val_running_accuracy/len(validation_loader.dataset) * 100:.4f} %')

In [ ]:
plt.plot(loss_history, label='Training Loss')
plt.plot(val_loss_history, label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
plt.plot(accuracy_history, label='Training Accuracy')
plt.plot(val_accuracy_history, label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
def im_convert(tensor):
    image = tensor.clone().detach().numpy()
    image = image.transpose(1, 2, 0)
    image = image * np.array((0.5, 0.5, 0.5))
    image = image.clip(image)
    return image

In [ ]:
images, labels = next(iter(training_loader))

fig = plt.figure(figsize=(25, 4))
for idx in range(20):
    ax = fig.add_subplot(2, 10, idx+1, xticks=[], yticks=[])
    plt.imshow(im_convert(images[idx]))
    ax.set_title(labels[idx].item())